In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, top_k_accuracy_score
from xgboost import XGBClassifier
import uvicorn
from fastapi import FastAPI, Request, File, UploadFile, HTTPException

In [6]:
# 3. Load datasets
train_df = pd.read_csv('Training.csv')
test_df = pd.read_csv('Testing.csv')

# Clean column headers of accidental spaces
train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

# Drop unnamed/empty trailing columns if present
train_df = train_df.loc[:, ~train_df.columns.str.contains('^Unnamed')]
test_df = test_df.loc[:, ~test_df.columns.str.contains('^Unnamed')]

# Deduplicate training samples to prevent data leakage
train_df = train_df.drop_duplicates()

# 4. Feature and Target Separation
TARGET_COL = 'prognosis'
X_train = train_df.drop(columns=[TARGET_COL])
y_train_raw = train_df[TARGET_COL]

X_test = test_df.drop(columns=[TARGET_COL])
y_test_raw = test_df[TARGET_COL]

# Preserving exact feature column ordering for API ingestion
feature_names = list(X_train.columns)

# 5. Label Encoding
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_test = label_encoder.transform(y_test_raw)

In [7]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42
)
model.fit(X_train, y_train)

# 7. Model Evaluation
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
top3_acc = top_k_accuracy_score(y_test, y_proba, k=3)

print(f"Test Accuracy:       {acc * 100:.2f}%")
print(f"Top-3 Test Accuracy: {top3_acc * 100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# 1. Setting the output directory to an 'artifacts' folder 
OUTPUT_DIR = 'artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Saving the model and label encoder locally
joblib.dump(model, os.path.join(OUTPUT_DIR, 'disease_model.pkl'))
joblib.dump(label_encoder, os.path.join(OUTPUT_DIR, 'label_encoder.pkl'))


# 3. Save the symptom feature names locally as a JSON file
with open(os.path.join(OUTPUT_DIR, 'symptom_features.json'), 'w') as f:
    json.dump(feature_names, f)

print(f"\nArtifacts successfully serialized and saved to local folder: {os.path.abspath(OUTPUT_DIR)}")

Test Accuracy:       97.62%
Top-3 Test Accuracy: 100.00%

Classification Report:

                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00         1
                                   AIDS       1.00      1.00      1.00         1
                                   Acne       1.00      1.00      1.00         1
                    Alcoholic hepatitis       1.00      1.00      1.00         1
                                Allergy       1.00      1.00      1.00         1
                              Arthritis       1.00      1.00      1.00         1
                       Bronchial Asthma       1.00      1.00      1.00         1
                   Cervical spondylosis       1.00      1.00      1.00         1
                            Chicken pox       0.50      1.00      0.67         1
                    Chronic cholestasis       1.00      1.00      1.00         1
                          

## Backend Development & Cloud AI 

### Creating Fast API Server

In [ ]:
from pydantic import BaseModel

app = FastAPI()

ARTIFACTS_DIR = 'artifacts'

try:
    model_pkl = joblib.load(os.path.join(ARTIFACTS_DIR, 'disease_model.pkl'))
    label_encoder = joblib.load(os.path.join(ARTIFACTS_DIR, 'label_encoder.pkl'))
except FileNotFoundError:
    raise RuntimeError("Model files not found")

class SymptomInput(BaseModel):
    symptoms: list[int]

with open(os.path.join(ARTIFACTS_DIR, 'symptom_features.json'), 'r') as f:
    expected_features = json.load(f)

@app.post("/predict")
async def predict_disease(payload: SymptomInput):
    symptoms_list = payload.symptoms

    input_features = np.array([symptoms_list])

    predicted_numeric_idx = model_pkl.predict(input_features)
    predict_disease = label_encoder.inverse_transform(predicted_numeric_idx)[0]

    probabilities = model_pkl.predict_proba(input_features)[0]
    risk_percentage = max(probabilities) * 100

    return {
        "disease": str(predict_disease),
        "risk_percentage": float(round(risk_percentage, 2))
    }

# Execution block
if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    
    config = uvicorn.Config(app, host="127.0.0.1", port=8000)
    server = uvicorn.Server(config)
    
    await server.serve()


INFO:     Started server process [12532]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:61566 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:61566 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:61566 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:60294 - "POST /predict HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [12532]


In [ ]:
from PIL import Image
import google.genai
from dotenv import load_dotenv

